In [ ]:
import os
import torch
from botorch import fit_gpytorch_mll
import botorch_tool
import copy
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.transforms import unnormalize, normalize, standardize
import numpy as np 
import pickle


torch.set_default_dtype(torch.float64)

from botorch.test_functions.multi_objective import BraninCurrin
tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}
problem = BraninCurrin(negate=True).to(**tkwargs)

ININITIAL_SAMPLING_NUM = 2

np.random.seed(42)
torch.manual_seed(42)

In [ ]:
data_total = []
for repeat in range(16):
    # initial random sampling
    train_observation_x = torch.cat([torch.rand(ININITIAL_SAMPLING_NUM, 2), 
                                     torch.randint(3, (ININITIAL_SAMPLING_NUM, 1))/2, 
                                     torch.randint(3, (ININITIAL_SAMPLING_NUM, 1))/2], dim=-1)
    train_observation_y = problem(train_observation_x[:,0:2])
    # begin the optimization loop 
    for budget in range(100):
        print(f"{repeat}_{budget} \r")
        # normalize training x to [0,1]
        train_x = train_observation_x
        # standarize train_y
        train_y = standardize(train_observation_y)
        # get models and train the models
        mll, model = botorch_tool.tools.initialize_noiseless_model(train_x = train_x, train_obj = train_y, cat_dims = [2,3])
        fit_gpytorch_mll(mll)
        # define the boundaries for the optimization problem 
        bounds = torch.stack([torch.zeros(train_x.shape[1]), torch.ones(train_x.shape[1])])
        categorical_values = {2: [0, 0.5, 1.0], 3: [0, 0.5, 1.0]}
        key_set = list(categorical_values.keys())
        value_set = list(categorical_values.values())
        fixed_features_list = []
        for i in value_set[0]:
            for j in value_set[1]:
                fixed_features_list.append({key_set[0] : i, key_set[1] : j})
        # get the candidate 
        MC_SAMPLES = 512
        sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))
        candidate = botorch_tool.tools.optimize_qnehvi_and_get_observation(
            model = model, 
            train_x = train_x, 
            ref_point = [0,0],
            sampler = sampler, 
            cat_dims = [2,3], 
            fixed_features_list = fixed_features_list)
        train_observation_x = torch.concat((train_observation_x,candidate))
        train_observation_y = torch.concat((train_observation_y,problem(candidate)))
        
    data_total.append(
        [train_observation_x,train_observation_y]
    )

In [ ]:
# read in data

In [ ]:
with open(f"./qnehvi.pkl", 'wb') as fp:
    pickle.dump(data_total, fp)

In [ ]:
import pickle

f = open('./qnehvi.pkl', 'rb')
data2 = pickle.load(f)  

f = open('./qehvi.pkl', 'rb')
data3 = pickle.load(f)  

f = open('./qehvi4_16.pkl', 'rb')
data4 = pickle.load(f)  

data3 = data3+data4

f = open('./gryffin.pkl', 'rb')
data4 = pickle.load(f)  

f = open('./gryffin2.pkl', 'rb')
data5 = pickle.load(f)

In [ ]:
from botorch.utils.multi_objective.box_decompositions.dominated import (
    DominatedPartitioning,
)
from botorch.utils.multi_objective.pareto import is_non_dominated

hvs_list_total = []
for i in range(16):
    train_observation_y = data2[i][1]
    hvs_list = []
    for i in range(len(train_observation_y)-ININITIAL_SAMPLING_NUM):
        bd = DominatedPartitioning(ref_point=problem.ref_point, 
                                   Y = train_observation_y[0:ININITIAL_SAMPLING_NUM+i])
        volume = bd.compute_hypervolume().item()
        hvs_list.append(volume)
    hvs_list_total.append(hvs_list)

hvs_list_total2 = []
for i in range(16):
    train_observation_y = data3[i][1]
    hvs_list = []
    for i in range(len(train_observation_y)-ININITIAL_SAMPLING_NUM):
        bd = DominatedPartitioning(ref_point=problem.ref_point, 
                                   Y = train_observation_y[0:ININITIAL_SAMPLING_NUM+i])
        volume = bd.compute_hypervolume().item()
        hvs_list.append(volume)
    hvs_list_total2.append(hvs_list)    
    
hvs_list_total3 = []
for i in range(16):
    train_observation_y = torch.tensor([list(data4[i][j].values())[-2:] for j in range(len(data4[i]))])
    hvs_list = []
    for i in range(len(train_observation_y)-ININITIAL_SAMPLING_NUM):
        bd = DominatedPartitioning(ref_point=problem.ref_point, 
                                   Y = train_observation_y[0:ININITIAL_SAMPLING_NUM+i])
        volume = bd.compute_hypervolume().item()
        hvs_list.append(volume)
    hvs_list_total3.append(hvs_list)    
    
hvs_list_total4 = []
for i in range(16):
    train_observation_y = torch.tensor([list(data5[i][j].values())[-2:] for j in range(len(data5[i]))])
    hvs_list = []
    for i in range(len(train_observation_y)-ININITIAL_SAMPLING_NUM):
        bd = DominatedPartitioning(ref_point=problem.ref_point, 
                                   Y = train_observation_y[0:ININITIAL_SAMPLING_NUM+i])
        volume = bd.compute_hypervolume().item()
        hvs_list.append(volume)
    hvs_list_total4.append(hvs_list)    

In [ ]:
from matplotlib import pyplot as plt
hvs_list_total = np.array(hvs_list_total)
hvs_list_total2 = np.array(hvs_list_total2)
hvs_list_total3 = np.array(hvs_list_total3)
hvs_list_total4 = np.array(hvs_list_total4)

plt.plot(hvs_list_total.mean(axis = 0), color = "blue", label = "qNEHI")
plt.fill_between(np.arange(len(hvs_list_total.mean(axis = 0))), 
                 hvs_list_total.mean(axis = 0) + hvs_list_total.std(axis = 0), 
                 hvs_list_total.mean(axis = 0) - hvs_list_total.std(axis = 0),
                 color='blue', alpha=0.5)

plt.plot(hvs_list_total2.mean(axis = 0), color = "red", label = "qEHI")
plt.fill_between(np.arange(len(hvs_list_total2.mean(axis = 0))), 
                 hvs_list_total2.mean(axis = 0) + hvs_list_total2.std(axis = 0), 
                 hvs_list_total2.mean(axis = 0) - hvs_list_total2.std(axis = 0),
                 color='red', alpha=0.5)

plt.plot(hvs_list_total3.mean(axis = 0), color = "black", label = "Gryffin + Chimera")
plt.fill_between(np.arange(len(hvs_list_total3.mean(axis = 0))), 
                 hvs_list_total3.mean(axis = 0) + hvs_list_total3.std(axis = 0), 
                 hvs_list_total3.mean(axis = 0) - hvs_list_total3.std(axis = 0),
                 color='black', alpha=0.5)

plt.plot(hvs_list_total4.mean(axis = 0), color = "green", label = "Gryffin2 + Chimera")
plt.fill_between(np.arange(len(hvs_list_total4.mean(axis = 0))), 
                 hvs_list_total4.mean(axis = 0) + hvs_list_total4.std(axis = 0), 
                 hvs_list_total4.mean(axis = 0) - hvs_list_total4.std(axis = 0),
                 color='black', alpha=0.5)

plt.axhline(problem.max_hv, color = "black")

plt.legend()

In [ ]:
hvs_list_total = np.log10(problem.max_hv - hvs_list_total)
hvs_list_total2 = np.log10(problem.max_hv - hvs_list_total2)
hvs_list_total3 = np.log10(problem.max_hv - hvs_list_total3)
hvs_list_total4 = np.log10(problem.max_hv - hvs_list_total4)

plt.plot(hvs_list_total.mean(axis = 0), color = "blue", label = "qNEHI")
plt.fill_between(np.arange(len(hvs_list_total.mean(axis = 0))), 
                 hvs_list_total.mean(axis = 0) + hvs_list_total.std(axis = 0), 
                 hvs_list_total.mean(axis = 0) - hvs_list_total.std(axis = 0),
                 color='blue', alpha=0.5)

plt.plot(hvs_list_total2.mean(axis = 0), color = "red", label = "qEHI")
plt.fill_between(np.arange(len(hvs_list_total2.mean(axis = 0))), 
                 hvs_list_total2.mean(axis = 0) + hvs_list_total2.std(axis = 0), 
                 hvs_list_total2.mean(axis = 0) - hvs_list_total2.std(axis = 0),
                 color='red', alpha=0.5)

plt.plot(hvs_list_total3.mean(axis = 0), color = "black", label = "Gryffin + Chimera")
plt.fill_between(np.arange(len(hvs_list_total3.mean(axis = 0))), 
                 hvs_list_total3.mean(axis = 0) + hvs_list_total3.std(axis = 0), 
                 hvs_list_total3.mean(axis = 0) - hvs_list_total3.std(axis = 0),
                 color='black', alpha=0.5)

plt.plot(hvs_list_total4.mean(axis = 0), color = "green", label = "Gryffin2 + Chimera")
plt.fill_between(np.arange(len(hvs_list_total4.mean(axis = 0))), 
                 hvs_list_total4.mean(axis = 0) + hvs_list_total4.std(axis = 0), 
                 hvs_list_total4.mean(axis = 0) - hvs_list_total4.std(axis = 0),
                 color='black', alpha=0.5)

plt.legend()